# Power Simulation Analysis

Standalone simulation notebook for the supplement power tables (eTables 7-14) and reviewer
response. This consolidates the power-simulation cells currently repeated in the DML notebooks.

Core simulation, matching the DML notebooks:

- simulate binary treatment `T`;
- simulate a latent responsive subgroup `G`;
- assign an oracle CATE score equal to the true subgroup treatment effect;
- simulate a binary outcome;
- test `Y ~ T + CATE` versus `Y ~ T + CATE + T:CATE` by likelihood-ratio test;
- estimate power as the proportion of simulations with interaction p < 0.05.

This notebook runs all four datasets and both outcomes. It writes raw and publication-formatted
CSV files under `pooled/power_simulation_outputs/`.

Run from `pooled/`:

```bash
N_SIMS=2000 python _build_power_simulation_analysis.py
jupyter nbconvert --to notebook --execute powerSimulationAnalysis.ipynb \
  --output powerSimulationAnalysis.executed.ipynb --ExecutePreprocessor.timeout=-1
```


In [1]:
# CONFIG
import glob
import os

SEED = int(os.environ.get('SEED', '42'))
N_SIMS = int(os.environ.get('N_SIMS', '2000'))
ALPHA = float(os.environ.get('ALPHA', '0.05'))

SUBGROUP_PREVALENCES = [0.20, 0.30, 0.50]
ABS_TREATMENT_EFFECTS = [0.10, 0.15, 0.20]

# Make direct execution from the repo root behave like the runner (`cd pooled`).
if os.path.basename(os.getcwd()) != 'pooled' and os.path.isdir('pooled'):
    os.chdir('pooled')

REPO_ROOT = os.path.abspath('..')
OUTDIR = os.path.abspath('power_simulation_outputs')
os.makedirs(OUTDIR, exist_ok=True)

CSV_CANDIDATES = {
    'eICU': [
        os.path.join(REPO_ROOT, 'eICU', 'eICUPredictorsDiag.csv'),
        os.path.join(REPO_ROOT, 'eICU', 'eICUPredictors*.csv'),
    ],
    'PMAP': [
        os.path.join(REPO_ROOT, 'pmap', 'PMAP_Predictors2.csv'),
        os.path.join(REPO_ROOT, 'pmap', 'PMAP_*2.csv'),
        os.path.join(REPO_ROOT, 'pmap', 'PMAP*Predictors*.csv'),
    ],
    'MIMIC-IV': [
        os.path.join(REPO_ROOT, 'mimiciv', 'MIMIC_Predictors.csv'),
        os.path.join(REPO_ROOT, 'mimiciv', 'MIMIC_*Predictors.csv'),
        os.path.join(REPO_ROOT, 'mimiciv', 'MIMIC*Predictors*.csv'),
    ],
    'HYPERION': [
        os.path.join(REPO_ROOT, 'hyperion', 'predictorsDf.csv'),
        os.path.join(REPO_ROOT, 'hyperion', '*predictors*.csv'),
    ],
}


def resolve_csv(name):
    matches = []
    for candidate in CSV_CANDIDATES[name]:
        matches.extend(sorted(glob.glob(candidate)))
    if not matches:
        return CSV_CANDIDATES[name][0]
    seen = set()
    ordered = []
    for p in matches:
        if p not in seen:
            ordered.append(p)
            seen.add(p)
    return ordered[0]


CSV = {name: resolve_csv(name) for name in CSV_CANDIDATES}
print('Resolved predictor CSVs:')
for name, path in CSV.items():
    print(f'  {name}: {path}')


Resolved predictor CSVs:
  eICU: /home/mbranda1/ttmhte/eICU/eICUPredictorsDiag.csv
  PMAP: /home/mbranda1/ttmhte/pmap/PMAP_Predictors2.csv
  MIMIC-IV: /home/mbranda1/ttmhte/mimiciv/MIMIC_Predictors.csv
  HYPERION: /home/mbranda1/ttmhte/hyperion/predictorsDf.csv


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.stats import chi2

pd.set_option('display.width', 180)
pd.set_option('display.max_columns', 80)


## Cohort Loaders

These loaders reproduce the cohort filters used in the DML notebooks and return canonical
columns: `TTM`, `mortality`, `neuro_favorable`.


In [3]:
UNSCORABLE = 'Unable to score due to medication'


def require_file(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f'Missing {path}. Restore predictor CSVs before executing.')


def load_eicu():
    require_file(CSV['eICU'])
    df = pd.read_csv(CSV['eICU'])
    f = (df['LastMGCS'] != UNSCORABLE) & (~df['LastMGCS'].isna())
    f = f & (df['FirstMGCSTime'] != df['LastMGCSTime'])
    for c in ['FirstGCS', 'FirstMGCS', 'LastMGCS', 'LastGCS']:
        if c in df.columns:
            df.loc[df[c] == UNSCORABLE, c] = np.nan
    df.loc[df['DeathAtDischarge'] == 1, 'LastMGCS'] = 1
    df.loc[f, 'LastMGCSPositive'] = (df.loc[f, 'LastMGCS'].astype(float) == 6).astype(int)
    df = df[f & (df['nurse_first_Motor'] != 6) & ~df['Hypothermia'].isna()].copy()
    return df.rename(columns={
        'Hypothermia': 'TTM',
        'DeathAtDischarge': 'mortality',
        'LastMGCSPositive': 'neuro_favorable',
    })


def _load_epic_style(name):
    require_file(CSV[name])
    df = pd.read_csv(CSV[name])
    f = (df['first_mGCS_time'] != df['last_mGCS_time'])
    df.loc[df['death_at_disch'] == 1, 'last_mGCS'] = 1
    df.loc[f, 'LastMGCSPositive'] = (df.loc[f, 'last_mGCS'].astype(float) == 6).astype(int)
    df = df[(df['first_mGCS'] != 6) & ~df['hypothermia'].isna()].copy()
    return df.rename(columns={
        'hypothermia': 'TTM',
        'death_at_disch': 'mortality',
        'LastMGCSPositive': 'neuro_favorable',
    })


def load_pmap():
    return _load_epic_style('PMAP')


def load_mimic():
    return _load_epic_style('MIMIC-IV')


def load_hyperion():
    require_file(CSV['HYPERION'])
    df = pd.read_csv(CSV['HYPERION'])
    df = df[df['groupe'] != 2].copy()
    df['TTM'] = (df['groupe'] == 1).astype(int)
    return df.rename(columns={
        'hospital_mortality': 'mortality',
        'CPC12': 'neuro_favorable',
    })


LOADERS = {
    'HYPERION': load_hyperion,
    'PMAP': load_pmap,
    'eICU': load_eicu,
    'MIMIC-IV': load_mimic,
}


def dataset_outcome_specs():
    rows = []
    for dataset, loader in LOADERS.items():
        try:
            df = loader()
        except Exception as e:
            rows.append({'dataset': dataset, 'status': f'missing: {e}'})
            continue
        for outcome_name, outcome_col in {'mortality': 'mortality', 'neuro': 'neuro_favorable'}.items():
            d = df[['TTM', outcome_col]].copy()
            d['TTM'] = pd.to_numeric(d['TTM'], errors='coerce')
            d[outcome_col] = pd.to_numeric(d[outcome_col], errors='coerce')
            d = d.dropna()
            untreated = d[d['TTM'] == 0]
            rows.append({
                'dataset': dataset,
                'outcome': outcome_name,
                'outcome_col': outcome_col,
                'n': int(len(d)),
                'n_ttm': int(d['TTM'].sum()),
                'treat_prev': float(d['TTM'].mean()),
                'baseline_risk': float(untreated[outcome_col].mean()) if len(untreated) else np.nan,
                'observed_outcome_rate': float(d[outcome_col].mean()) if len(d) else np.nan,
                'status': 'ok',
            })
    return pd.DataFrame(rows)


specs = dataset_outcome_specs()
specs.to_csv(os.path.join(OUTDIR, 'dataset_outcome_specs.csv'), index=False)
display(specs)


/local/mbranda1/3981445/ipykernel_4073179/2703325627.py:11: DtypeWarning: Columns (2188,2190) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV['eICU'])


,dataset,outcome,outcome_col,n,n_ttm,treat_prev,baseline_risk,observed_outcome_rate,status
0,HYPERION,mortality,mortality,581,284,0.488812,0.831650,0.817556,ok
1,HYPERION,neuro,neuro_favorable,581,284,0.488812,0.057239,0.077453,ok
2,PMAP,mortality,mortality,1412,666,0.471671,0.585791,0.660057,ok
3,PMAP,neuro,neuro_favorable,1289,619,0.480217,0.319403,0.245151,ok
4,eICU,mortality,mortality,1842,608,0.330076,0.516207,0.542888,ok
5,eICU,neuro,neuro_favorable,1842,608,0.330076,0.368720,0.337676,ok
6,MIMIC-IV,mortality,mortality,611,280,0.458265,0.670695,0.641571,ok
7,MIMIC-IV,neuro,neuro_favorable,560,280,0.500000,0.335714,0.326786,ok


## DML Power-Simulation Functions

In [4]:
def simulate_dataset_hte(
    n,
    treat_prev,
    subgroup_prev,
    baseline_risk,
    abs_treatment_effect=0.10,
    rng=None,
):
    if rng is None:
        rng = np.random.default_rng()

    T = rng.binomial(1, treat_prev, size=n)
    G = rng.binomial(1, subgroup_prev, size=n)
    true_tau = abs_treatment_effect * G
    p = baseline_risk + T * true_tau
    p = np.clip(p, 1e-6, 1 - 1e-6)
    Y = rng.binomial(1, p, size=n)
    cate_score = true_tau
    return pd.DataFrame({'T': T, 'G': G, 'cate_score': cate_score, 'Y': Y})


def lr_test_interaction(df):
    X_reduced = pd.DataFrame({
        'const': 1.0,
        'T': df['T'],
        'cate_score': df['cate_score'],
    })
    X_full = pd.DataFrame({
        'const': 1.0,
        'T': df['T'],
        'cate_score': df['cate_score'],
        'interaction': df['T'] * df['cate_score'],
    })
    try:
        m0 = sm.Logit(df['Y'], X_reduced).fit(disp=False)
        m1 = sm.Logit(df['Y'], X_full).fit(disp=False)
        lr_stat = 2 * (m1.llf - m0.llf)
        pval = chi2.sf(lr_stat, X_full.shape[1] - X_reduced.shape[1])
        return {'lr_stat': lr_stat, 'pval': pval, 'converged': True}
    except Exception:
        return {'lr_stat': np.nan, 'pval': np.nan, 'converged': False}


def estimate_power_for_scenario(
    n,
    treat_prev,
    subgroup_prev,
    baseline_risk,
    abs_treatment_effect=0.10,
    n_sims=N_SIMS,
    alpha=ALPHA,
    seed=SEED,
):
    rng = np.random.default_rng(seed)
    pvals = []
    n_fail = 0
    for _ in range(n_sims):
        df = simulate_dataset_hte(
            n=n,
            treat_prev=treat_prev,
            subgroup_prev=subgroup_prev,
            baseline_risk=baseline_risk,
            abs_treatment_effect=abs_treatment_effect,
            rng=rng,
        )
        res = lr_test_interaction(df)
        if res['converged'] and np.isfinite(res['pval']):
            pvals.append(res['pval'])
        else:
            n_fail += 1
    pvals = np.asarray(pvals)
    return {
        'n': n,
        'treat_prev': treat_prev,
        'subgroup_prev': subgroup_prev,
        'baseline_risk': baseline_risk,
        'abs_treatment_effect': abs_treatment_effect,
        'n_sims': n_sims,
        'n_successful_fits': int(len(pvals)),
        'n_failed_fits': int(n_fail),
        'power': float(np.mean(pvals < alpha)) if len(pvals) else np.nan,
    }


## Run All Scenarios

In [5]:
results = []
ok_specs = specs[specs['status'].eq('ok')].copy()
for _, spec in ok_specs.iterrows():
    for subgroup_prev in SUBGROUP_PREVALENCES:
        for abs_te in ABS_TREATMENT_EFFECTS:
            out = estimate_power_for_scenario(
                n=int(spec['n']),
                treat_prev=float(spec['treat_prev']),
                subgroup_prev=float(subgroup_prev),
                baseline_risk=float(spec['baseline_risk']),
                abs_treatment_effect=float(abs_te),
                n_sims=N_SIMS,
                alpha=ALPHA,
                seed=SEED,
            )
            out['dataset'] = spec['dataset']
            out['outcome'] = spec['outcome']
            results.append(out)
            print(f"{spec['dataset']:9s} {spec['outcome']:9s} subgroup={subgroup_prev:.2f} effect={abs_te:.2f} power={out['power']:.3f}")

power_table = pd.DataFrame(results)
if power_table.empty:
    power_table = pd.DataFrame(columns=[
        'dataset', 'outcome', 'n', 'treat_prev', 'subgroup_prev', 'baseline_risk',
        'abs_treatment_effect', 'n_sims', 'n_successful_fits', 'n_failed_fits', 'power',
    ])
power_table = power_table[[
    'dataset', 'outcome', 'n', 'treat_prev', 'subgroup_prev', 'baseline_risk',
    'abs_treatment_effect', 'n_sims', 'n_successful_fits', 'n_failed_fits', 'power',
]]
power_table.to_csv(os.path.join(OUTDIR, 'power_simulation_raw.csv'), index=False)
display(power_table)


/home/mbranda1/.local/lib/python3.11/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/home/mbranda1/.local/lib/python3.11/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/home/mbranda1/.local/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/home/mbranda1/.local/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/home/mbranda1/.local/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  w

HYPERION  mortality subgroup=0.20 effect=0.10 power=0.359


/home/mbranda1/.local/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/home/mbranda1/.local/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/home/mbranda1/.local/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/home/mbranda1/.local/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/home/mbranda1/.local/lib/python3.11/site-packages/statsmodels/base/

## Supplement-Formatted Tables

In [ ]:
paper = power_table.copy()
paper = paper.rename(columns={
    'dataset': 'Dataset',
    'outcome': 'Outcome',
    'n': 'N',
    'treat_prev': 'TTM prevalence',
    'subgroup_prev': 'Responsive subgroup prevalence (%)',
    'baseline_risk': 'Baseline risk',
    'abs_treatment_effect': 'Assumed absolute treatment effect',
    'n_sims': 'Simulations',
    'n_successful_fits': 'Successful fits',
    'n_failed_fits': 'Failed fits',
    'power': 'Power',
})
paper['Responsive subgroup prevalence (%)'] = (paper['Responsive subgroup prevalence (%)'] * 100).astype(int)
paper['TTM prevalence'] = paper['TTM prevalence'].round(3)
paper['Baseline risk'] = paper['Baseline risk'].round(3)
paper['Assumed absolute treatment effect'] = paper['Assumed absolute treatment effect'].round(2)
paper['Power'] = paper['Power'].round(3)
paper = paper[[
    'Dataset', 'Outcome', 'N', 'TTM prevalence', 'Baseline risk',
    'Responsive subgroup prevalence (%)', 'Assumed absolute treatment effect',
    'Simulations', 'Successful fits', 'Failed fits', 'Power',
]]
paper.to_csv(os.path.join(OUTDIR, 'power_simulation_paper_tables.csv'), index=False)

for (dataset, outcome), sub in paper.groupby(['Dataset', 'Outcome'], sort=False):
    path = os.path.join(OUTDIR, f"power_{dataset.lower().replace('-', '').replace(' ', '_')}_{outcome}.csv")
    sub.to_csv(path, index=False)

display(paper)


## Figures and Paste-Ready Summary

In [ ]:
def plot_power(power_table, outcome):
    sub = power_table[power_table['outcome'].eq(outcome)].copy()
    if sub.empty:
        print(f'No power rows for {outcome}; skipping plot.')
        return
    fig, axes = plt.subplots(1, len(ABS_TREATMENT_EFFECTS), figsize=(15, 4), sharey=True)
    for ax, effect in zip(axes, ABS_TREATMENT_EFFECTS):
        s = sub[np.isclose(pd.to_numeric(sub['abs_treatment_effect'], errors='coerce'), effect)]
        for dataset, d in s.groupby('dataset', sort=False):
            d = d.sort_values('subgroup_prev')
            ax.plot(d['subgroup_prev'] * 100, d['power'], marker='o', label=dataset)
        ax.axhline(0.80, color='0.5', ls='--', lw=1)
        ax.set_title(f'Absolute effect = {effect:.0%}')
        ax.set_xlabel('Responsive subgroup prevalence (%)')
        ax.set_ylim(0, 1)
    axes[0].set_ylabel('Detection probability')
    axes[-1].legend(loc='lower right')
    fig.suptitle(f'Simulation-based HTE power: {outcome}')
    fig.tight_layout()
    path = os.path.join(OUTDIR, f'power_curves_{outcome}.png')
    fig.savefig(path, dpi=180)
    plt.show()


plot_power(power_table, 'mortality')
plot_power(power_table, 'neuro')

print('--- Paste-ready power summary ---')
focus = power_table[(power_table['subgroup_prev'].eq(0.20)) & (power_table['abs_treatment_effect'].isin([0.10, 0.20]))]
for outcome, sub in focus.groupby('outcome', sort=False):
    p10 = sub[sub['abs_treatment_effect'].eq(0.10)]['power']
    p20 = sub[sub['abs_treatment_effect'].eq(0.20)]['power']
    print(f"{outcome}: 10-point effect in 20% subgroup power range {p10.min():.2f}-{p10.max():.2f}; "
          f"20-point effect in 20% subgroup power range {p20.min():.2f}-{p20.max():.2f}.")
